In [16]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
import soundfile as sf
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "quickstart") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

from inference.rt import run_inference


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
def run_experiment_and_save(model_path, checkpoint_name, offline_parameters=None, output_path=None, conditioning_sequence=None, offline_duration=10.0, cascade_mode="hard"):
    if output_path is None:
        raise ValueError("output_path is required")

    if offline_parameters is not None and conditioning_sequence is not None:
        raise ValueError("Pass either offline_parameters or conditioning_sequence, not both")

    if offline_parameters is None and conditioning_sequence is None:
        raise ValueError("Pass offline_parameters for static conditioning or conditioning_sequence for time-varying conditioning")

    output_path = pathlib.Path(output_path)
    if output_path.suffix.lower() != ".wav":
        output_path = output_path.with_suffix(".wav")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    audio = run_inference(
        model_dir=model_path,
        checkpoint_name=checkpoint_name,
        mode="offline",
        cascade_mode=cascade_mode,
        offline_duration=offline_duration,
        offline_params=offline_parameters,
        conditioning_sequence=conditioning_sequence,
    )

    sf.write(output_path, audio, 24000, subtype="PCM_16")
    print(f"Saved WAV to: {output_path}")
    return output_path


In [18]:

cont_path      = "output_26.02.28.18.08" # Path to save the trained model and configs
class_path = "output_class_26.03.12.11.16"
film_path = "output_Film_26.03.12.11.16"
checkpoint_name = "last_checkpoint.pt"   # Checkpoint to use for inference (e.g., "last_checkpoint.pt" or "checkpoint_75.pt")


In [19]:
# Use the same note across all three models.
# For the continuous models, that means a pitch in Hz.
# For the class model, that means a one-hot note-class vector.
note_name = "A"
note_hz = 440.0  # A4, inside the 230-450 Hz conditioning range
offline_duration = 5.0

cont_offline_params = {
    "pitch": note_hz,
}

pitch_classes = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
class_offline_params = {
    f"class_{pitch_class}": 1.0 if pitch_class == note_name else 0.0
    for pitch_class in pitch_classes
}

film_offline_params = {
    "pitch": note_hz,
}

print("cont_offline_params:", cont_offline_params)
print("film_offline_params:", film_offline_params)
print("class_offline_params:", class_offline_params)

run_experiment_and_save(
    cont_path,
    checkpoint_name,
    cont_offline_params,
    output_path=f"./experiments_output/cont_{note_name}",
    offline_duration=offline_duration,
)

run_experiment_and_save(
    film_path,
    checkpoint_name,
    film_offline_params,
    output_path=f"./experiments_output/film_{note_name}",
    offline_duration=offline_duration,
)

run_experiment_and_save(
    class_path,
    checkpoint_name,
    class_offline_params,
    output_path=f"./experiments_output/class_{note_name}",
    offline_duration=offline_duration,
)


cont_offline_params: {'pitch': 440.0}
film_offline_params: {'pitch': 440.0}
class_offline_params: {'class_C': 0.0, 'class_C#': 0.0, 'class_D': 0.0, 'class_D#': 0.0, 'class_E': 0.0, 'class_F': 0.0, 'class_F#': 0.0, 'class_G': 0.0, 'class_G#': 0.0, 'class_A': 1.0, 'class_A#': 0.0, 'class_B': 0.0}
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Generating 5.0s of audio (375 frames)...
Conditioning values:
  - pitch: 440.00 Hz

Generation complete!
  - Audio duration: 5.00s
  - Generation time: 1.01s
  - Real-time factor: 4.93x
Saved WAV to: experiments_output/cont_A.wav
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 128 of the GRU input size of 128
FiLM conditioning enabled on GRU input
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Generating 5.0s of audio (375 frames)...
Conditioning values:
  - pitch: 440.00 Hz

Generation complete!
  - Audio duration: 5.00s
  - Generation time: 0.94s
  - Real-time factor: 5.34x
Saved WAV to: experiments_output/film_A.wav
Loaded conditioning config with 12 features:
  - class_C: binary
  - class_C#: binary
  - class_D: binary
  - class_D#: binary
  - class_E: binary
  - class_F: binary
  - class_F#: binary
  - class_G: binary
  - class_G#: binary
  - class_A: binary
  - c

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 12
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Generating 5.0s of audio (375 frames)...
Conditioning values:
  - class_C: 0.00 probability
  - class_C#: 0.00 probability
  - class_D: 0.00 probability
  - class_D#: 0.00 probability
  - class_E: 0.00 probability
  - class_F: 0.00 probability
  - class_F#: 0.00 probability
  - class_G: 0.00 probability
  - class_G#: 0.00 probability
  - class_A: 1.00 probability
  - class_A#: 0.00 probability
  - class_B: 0.00 probability

Generation complete!
  - Au

PosixPath('experiments_output/class_A.wav')

In [20]:
# Time-varying experiment: play C for 2 seconds, then switch to D.
import torch
from inference.rt import FRAME_RATE, ParameterScaler, load_conditioning_config

def make_continuous_sequence(value, duration_seconds, scaler, feature_name="pitch"):
    num_frames = int(duration_seconds * FRAME_RATE)
    sequence = torch.zeros(num_frames, 1, dtype=torch.float32)
    sequence[:, 0] = scaler.normalize(feature_name, value)
    return sequence

def make_class_sequence(class_name, duration_seconds, feature_names):
    num_frames = int(duration_seconds * FRAME_RATE)
    sequence = torch.zeros(num_frames, len(feature_names), dtype=torch.float32)
    feature_name = f"class_{class_name}"
    class_idx = feature_names.index(feature_name)
    sequence[:, class_idx] = 1.0
    return sequence

segment_duration = 2.0

c_hz = 261.63  # C4
d_hz = 293.66  # D4

cont_scaler = ParameterScaler(load_conditioning_config(cont_path))
film_scaler = ParameterScaler(load_conditioning_config(film_path))
class_config = load_conditioning_config(class_path)
class_feature_names = class_config["feature_names"]



Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz
Loaded conditioning config with 12 features:
  - class_C: binary
  - class_C#: binary
  - class_D: binary
  - class_D#: binary
  - class_E: binary
  - class_F: binary
  - class_F#: binary
  - class_G: binary
  - class_G#: binary
  - class_A: binary
  - class_A#: binary
  - class_B: binary


In [21]:

cont_conditioning_sequence = torch.cat([
    make_continuous_sequence(c_hz, segment_duration, cont_scaler),
    make_continuous_sequence(d_hz, segment_duration, cont_scaler),
], dim=0)

film_conditioning_sequence = torch.cat([
    make_continuous_sequence(c_hz, segment_duration, film_scaler),
    make_continuous_sequence(d_hz, segment_duration, film_scaler),
], dim=0)

class_conditioning_sequence = torch.cat([
    make_class_sequence("C", segment_duration, class_feature_names),
    make_class_sequence("D", segment_duration, class_feature_names),
], dim=0)

print("cont_conditioning_sequence:", cont_conditioning_sequence.shape)
print("film_conditioning_sequence:", film_conditioning_sequence.shape)
print("class_conditioning_sequence:", class_conditioning_sequence.shape)

run_experiment_and_save(
    cont_path,
    checkpoint_name,
    conditioning_sequence=cont_conditioning_sequence,
    output_path="./experiments_output/cont_C_to_D",
)

run_experiment_and_save(
    film_path,
    checkpoint_name,
    conditioning_sequence=film_conditioning_sequence,
    output_path="./experiments_output/film_C_to_D",
)

run_experiment_and_save(
    class_path,
    checkpoint_name,
    conditioning_sequence=class_conditioning_sequence,
    output_path="./experiments_output/class_C_to_D",
)

cont_conditioning_sequence: torch.Size([300, 1])
film_conditioning_sequence: torch.Size([300, 1])
class_conditioning_sequence: torch.Size([300, 12])
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Using provided conditioning sequence: torch.Size([300, 1])

Generation complete!
  - Audio duration: 4.00s
  - Generation time: 0.77s
  - Real-time factor: 5.20x
Saved WAV to: experiments_output/cont_C_to_D.wav
Loaded conditioning config with 1 features:
  - pitch: 230-450 Hz

Loading EnCodec model...


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 128 of the GRU input size of 128
FiLM conditioning enabled on GRU input
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 1
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Using provided conditioning sequence: torch.Size([300, 1])

Generation complete!
  - Audio duration: 4.00s
  - Generation time: 0.76s
  - Real-time factor: 5.27x
Saved WAV to: experiments_output/film_C_to_D.wav
Loaded conditioning config with 12 features:
  - class_C: binary
  - class_C#: binary
  - class_D: binary
  - class_D#: binary
  - class_E: binary
  - class_F: binary
  - class_F#: binary
  - class_G: binary
  - class_G#: binary
  - class_A: binary
  - class_A#: binary
  -

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

Using checkpoint: last_checkpoint.pt
⚠️  Overriding cascade mode: hard → hard
⚠️  Overriding temperature: 0.8
⚠️  Overriding top-k: 8
Loading RNN model from last_checkpoint.pt...
Initializing the RNNGeneratorSoft on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model loaded successfully!
  - Hidden size: 128
  - Num layers: 3
  - Conditioning size: 12
  - Cascade mode: hard
  - Hard sampling mode: sample
  - Temperature: 0.8
  - Top-k: 8
  - Device: cpu
Using provided conditioning sequence: torch.Size([300, 12])

Generation complete!
  - Audio duration: 4.00s
  - Generation time: 0.70s
  - Real-time factor: 5.69x
Saved WAV to: experiments_output/class_C_to_D.wav


PosixPath('experiments_output/class_C_to_D.wav')